# Meteora DLMM Bin Atlas: connecting, fetching, normalizing, visualizing

This notebook explores Meteora DLMM bin-level liquidity using a read-only Solana RPC connection. It does not run a Solana node, does not use a wallet, and does not submit transactions.

## Environment

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from urllib.parse import urlparse
from dotenv import load_dotenv

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
PLOTS_DIR = PROJECT_ROOT / "plots"

load_dotenv(PROJECT_ROOT / ".env")


def latest_matching(directory: Path, pattern: str) -> Path:
    matches = sorted(directory.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern} in {directory}")
    return matches[-1]


print(f"Project root: {PROJECT_ROOT}")

## RPC model

We connect through an RPC provider using `SOLANA_RPC_URL`. This is read-only. We never print the full URL (it may contain API keys).

In [ ]:
rpc_url = os.getenv("SOLANA_RPC_URL", "")
rpc_host = urlparse(rpc_url).hostname or "(not set)"
cluster = os.getenv("SOLANA_CLUSTER", "mainnet-beta")

print(f"RPC host: {rpc_host}")
print(f"Cluster: {cluster}")

## Fetch pool candidates

Run the TypeScript discovery script, then load the processed CSV.

In [ ]:
!cd {PROJECT_ROOT} && npm run discover:pools

In [ ]:
pool_candidates_path = DATA_PROCESSED / "pool_candidates.csv"
pool_candidates = pd.read_csv(pool_candidates_path)
pool_candidates.head()

## Choose one pool

Use the first discovered candidate, or set `POOL_ADDRESS` manually.

In [ ]:
POOL_ADDRESS = pool_candidates.iloc[0]["pool_address"]
# POOL_ADDRESS = "5rCf1DM8LjKTw4YqhnoLcngyZYeNnQqztScTogYHAS6"

selected = pool_candidates[pool_candidates["pool_address"] == POOL_ADDRESS].iloc[0]
print(f"Pool: {POOL_ADDRESS}")
print(f"Label: {selected.get('raw_name_or_symbol_if_available', '')}")
print(f"Bin step: {selected.get('bin_step', '')}")

## Fetch pool snapshot

In [ ]:
!cd {PROJECT_ROOT} && npm run fetch:pool -- --pool {POOL_ADDRESS}

## Fetch bin arrays

In [ ]:
!cd {PROJECT_ROOT} && npm run fetch:bins -- --pool {POOL_ADDRESS}

## Normalize bin atlas

In [ ]:
!cd {PROJECT_ROOT} && npm run normalize:bins -- --pool {POOL_ADDRESS}

## Load bin atlas

In [ ]:
latest_bin_atlas_path = latest_matching(
    DATA_PROCESSED, f"bin_atlas_{POOL_ADDRESS}_*.csv"
)
df = pd.read_csv(latest_bin_atlas_path)
print(f"Loaded: {latest_bin_atlas_path.name}")
df.head()

In [ ]:
df.info()

## Basic sanity checks

In [ ]:
print("bin_id range:", df["bin_id"].min(), df["bin_id"].max())
print(
    "distance_from_active range:",
    df["distance_from_active"].min(),
    df["distance_from_active"].max(),
)
print("active bin rows:", df["is_active_bin"].astype(str).str.lower().eq("true").sum())

## Visualize liquidity by bin

In [ ]:
plot_df = df.sort_values("bin_id").copy()
plot_df["liquidity"] = pd.to_numeric(plot_df["liquidity"], errors="coerce").fillna(0)

plt.figure(figsize=(12, 5))
plt.bar(plot_df["distance_from_active"], plot_df["liquidity"])
plt.axvline(0, linestyle="--")
plt.xlabel("Distance from active bin")
plt.ylabel("Liquidity")
plt.title("Meteora DLMM liquidity by bin")
plt.tight_layout()
plt.show()

## Visualize token composition

If x/y amounts exist, plot them around the active bin. Amounts are raw on-chain units (not decimal-adjusted).

In [ ]:
plot_df["x_amount"] = pd.to_numeric(plot_df["x_amount"], errors="coerce").fillna(0)
plot_df["y_amount"] = pd.to_numeric(plot_df["y_amount"], errors="coerce").fillna(0)

plt.figure(figsize=(12, 5))
plt.plot(plot_df["distance_from_active"], plot_df["x_amount"], label="X amount")
plt.plot(plot_df["distance_from_active"], plot_df["y_amount"], label="Y amount")
plt.axvline(0, linestyle="--")
plt.xlabel("Distance from active bin")
plt.ylabel("Amount")
plt.title("Token composition around active bin")
plt.legend()
plt.tight_layout()
plt.show()

## Microstructure notes

The active bin functions as a local coordinate center. Liquidity away from the active bin is a discrete field over price bins. A single snapshot gives the static shape; repeated snapshots would show migration and deformation.

## Next questions

- How concentrated is liquidity around the active bin?
- How asymmetric is liquidity on either side?
- How does the active bin move over time?
- Do volatile pools show wider liquidity distributions?
- How do stable pools differ from memecoin or SOL pairs?